### Import dependencies

In [8]:
from pathlib import Path
from collections import deque
import cv2
import numpy as np

### Settings

In [9]:
THRESHOLD = 8
MIN_AREA = 150
GROUP_CLOSE_SIZE = 21
GROUP_DILATE_SIZE = 3
BOX_PAD = 4
# MERGE_DISTANCE = 50

### Preprocessing frames
Enhance contrast and denoise

In [10]:
def preprocess_frame(frame, clahe=None):
    """
    Convert frame into a grayscale image used for thresholding.
    For now this preserves your current behavior.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if clahe is not None:
        gray = clahe.apply(gray)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    return gray

### Thresholding
Separate objects from background by converting grayscale into binary images

In [11]:
def make_threshold_mask(gray, threshold=THRESHOLD):
    """
    Convert preprocessed grayscale frame into binary mask.
    """
    _, thresh = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    return thresh

### Calculating confidence score 
Compute confidence score for each detection (needed for tracking later). Confidence score is calculated using foreground area inside box, brightness of foreground pixels, contrast above background, box size, fill ratio.

In [12]:
def clamp(value, low=0.0, high=1.0):
    return max(low, min(high, value))


def compute_detection_score(gray, mask, box, real_area):
    """
    Compute synthetic objectness confidence for an OpenCV detection.

    This is not species confidence.
    It is confidence that this box is a real organism-like object.
    """
    x, y, w, h = box

    roi_gray = gray[y:y + h, x:x + w]
    roi_mask = mask[y:y + h, x:x + w] > 0

    if roi_gray.size == 0 or roi_mask.sum() == 0:
        return 0.0

    fg_values = roi_gray[roi_mask]

    mean_fg = float(np.mean(fg_values))
    p90_fg = float(np.percentile(fg_values, 90))
    max_fg = float(np.max(fg_values))

    box_area = w * h
    fill_ratio = real_area / max(box_area, 1)

    # Area score: favors larger trackable organisms.
    # Tune 2000 depending on your image size.
    area_score = clamp(real_area / 2000.0)

    # Brightness score: assumes CLAHE/gray values in 0-255.
    brightness_score = clamp(p90_fg / 80.0)

    # Fill score: fragmented organisms may have low fill, so keep this weak.
    fill_score = clamp(fill_ratio / 0.15)

    # Combined confidence.
    # Since you care about larger organisms, area gets the most weight.
    score = (
        0.45 * area_score +
        0.40 * brightness_score +
        0.15 * fill_score
    )

    return float(clamp(score))

### Contouring
Identify contour (bounding) boxes around objects

In [13]:
def get_grouped_boxes(mask, gray, min_area=MIN_AREA):
    """
    Merge fragmented bright regions into one box per likely organism.

    mask = original binary threshold mask
    group_mask = expanded mask used only for grouping fragments
    """
    # 1. Create grouping mask
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(GROUP_CLOSE_SIZE, GROUP_CLOSE_SIZE))
    dilate_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(GROUP_DILATE_SIZE, GROUP_DILATE_SIZE))

    group_mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,close_kernel,iterations=1)
    group_mask = cv2.dilate(group_mask,dilate_kernel,iterations=1)

    # 2. Find contours on grouped mask
    contours, _ = cv2.findContours(group_mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for contour in contours:
        gx, gy, gw, gh = cv2.boundingRect(contour)
        # Crop original mask inside grouped region
        original_roi = mask[gy:gy + gh, gx:gx + gw]
        # Use original foreground pixels, not the dilated pixels
        ys, xs = np.where(original_roi > 0)
        if len(xs) == 0:
            continue
        real_area = len(xs)
        if real_area < min_area:
            continue
        # Tight box around original foreground pixels
        x1 = gx + xs.min()
        y1 = gy + ys.min()
        x2 = gx + xs.max() + 1
        y2 = gy + ys.max() + 1
        # Add small padding
        x1 = max(0, x1 - BOX_PAD)
        y1 = max(0, y1 - BOX_PAD)
        x2 = min(mask.shape[1], x2 + BOX_PAD)
        y2 = min(mask.shape[0], y2 + BOX_PAD)
        w = x2 - x1
        h = y2 - y1

        box = (x1, y1, x2 - x1, y2 - y1)
        score = compute_detection_score(gray=gray, mask=mask, box=box, real_area=real_area)
        boxes.append({"box": box, "area": real_area, "score": score, "class_id": 0})

    return boxes, group_mask

Draw boxes around object based on contour

In [14]:
def draw_debug_boxes(frame, boxes, color=(0, 255, 0), thickness=2):
    debug_frame = frame.copy()

    for item in boxes:
        x, y, w, h = item["box"]
        score = item.get("score", 0)

        cv2.rectangle(
            debug_frame,
            (x, y),
            (x + w, y + h),
            color,
            thickness
        )

        cv2.putText(
            debug_frame,
            f"{score:.2f}",
            (x, max(0, y - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            1,
            cv2.LINE_AA
        )

    return debug_frame

### Video processing

In [15]:
def make_unique_output_folder(base_output_folder, video_stem):
    """
    Create a unique output folder for each run even on the same video
    """
    base_output_folder = Path(base_output_folder)
    candidate = base_output_folder / video_stem
    if not candidate.exists():
        candidate.mkdir(parents=True, exist_ok=False)
        return candidate
    counter = 1

    while True:
        candidate = base_output_folder / f"{video_stem}_{counter}"
        if not candidate.exists():
            candidate.mkdir(parents=True, exist_ok=False)
            return candidate
        counter += 1

In [19]:
def process_video(video_path, output_folder):
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))

    video_output_folder = make_unique_output_folder(output_folder, video_path.stem)

    debug_folder = Path("debug") / video_output_folder.name
    debug_folder.mkdir(parents=True, exist_ok=True)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

    frame_index = 0

    print(f"\nProcessing: {video_path.name}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_index==0:
            height, width = frame.shape[:2]
            print(f"Frame size: {width} * {height}")
        gray = preprocess_frame(frame, clahe=clahe)
        mask = make_threshold_mask(gray)
        boxes, group_mask = get_grouped_boxes(mask, gray)
        if len(boxes) > 0:
            debug_frame = draw_debug_boxes(frame, boxes)
            cv2.imwrite(str(debug_folder / f"debug_{frame_index:06d}.jpg"), debug_frame)

            #cv2.imwrite(str(debug_folder / f"group_mask_{frame_index:06d}.png"),group_mask)
        frame_index += 1

    cap.release()
    print(f"Finished: {video_path.name}")

### Call functions on a video

In [20]:
process_video("test_converted_videos/10.0.12.2_202307070748.mp4",'debug')


Processing: 10.0.12.2_202307070748.mp4
Frame size: 1920 * 1080
Finished: 10.0.12.2_202307070748.mp4
